
I want to create a ML model dataset of every (NACE, neighborhood) pair and a binary target based on if the specific area is top3 for this NACE

## ***Initials***

In [76]:
import pandas as pd
from itertools import product
import joblib
from sklearn.preprocessing import LabelEncoder

## ***Help Functions***

In [56]:
def is_top3(nace, neighborhood, top3_lookup):
    top3 = top3_lookup.get(nace)
    if not top3:
        return 0
    return int(neighborhood in top3.values())

## ***Load the Data***

In [57]:
top3_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\4. Ensembling of Approaches\\Extracted CSV Files\\top3_final.csv")
top3_df.columns = ["NACE_Code", "Top1", "Top2", "Top3"]
print(top3_df.shape)
print(top3_df.columns)
top3_df.head()

(306, 4)
Index(['NACE_Code', 'Top1', 'Top2', 'Top3'], dtype='object')


,NACE_Code,Top1,Top2,Top3
0,1.13,Metamorfosi,Kato Lehonia,Agios Konstantinos
1,1.30,Epta Platania - Oksigono,Agia Paraskeui,Agioi Anargiroi
2,1.49,Nees Pagases,Dimini,Melissatika
3,1.61,Epta Platania - Oksigono,Nees Pagases,Aivaliotika
4,2.40,Analipsi,Metamorfosi,Agios Konstantinos


In [58]:
neighborhoods_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\3. Base Datasets\\3. Data -  Smaller Spatial Units\\1. Neighborhoods\\Extracted CSV Files\\neighborhoods_enriched.csv")  # neighborhood-level features
neighborhoods_df.rename(columns={"Municipal Community": "Municipal_Community"}, inplace=True)
print(neighborhoods_df.shape)
print(neighborhoods_df.columns)
neighborhoods_df.head()

(48, 12)
Index(['Municipal_Community', 'Neighborhood_Greek', 'Neighborhood',
       'Centroid_x', 'Centroid_y', 'Neighborhood_Area_km2', 'Geometry',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')


,Municipal_Community,Neighborhood_Greek,Neighborhood,Centroid_x,Centroid_y,Neighborhood_Area_km2,Geometry,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km
0,Municipal Community of Dimini,Δημίνι,Dimini,22.882945,39.349112,37.344776,MULTIPOLYGON (((406692.718139543 4357521.85294...,6.154015,5.418577,0.443434,1.569912,4.273031
1,Municipal Community of Sesklos,Χρυσή Ακτή Παναγίας,Xrisi Akti Panagias,22.836899,39.305430,9.983032,MULTIPOLYGON (((396744.165394512 4351195.65810...,11.932475,11.015584,2.299921,2.410375,10.100771
2,Municipal Community of Sesklos,Σέσκλο,Sesklo,22.838168,39.353051,27.367312,MULTIPOLYGON (((400839.361742302 4353031.79092...,9.814666,9.191001,0.482308,4.965561,7.989610
3,Municipal Community of Volos,Άγιοι Ανάργυροι,Agioi Anargiroi,22.924059,39.366937,0.774436,MULTIPOLYGON (((406589.250339913 4357224.83344...,2.296184,1.978856,0.058142,0.063879,0.866021
4,Municipal Community of Volos,Αϊβαλιώτικα,Aivaliotika,22.922660,39.343696,4.848904,MULTIPOLYGON (((407619.510540358 4356306.42684...,3.508166,2.529372,0.224486,0.246542,1.913456


In [59]:
municipal_communities_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\3. Base Datasets\\2. Data - Municipal Communities\\4. Exploratory Data Analysis\\Extracted CSV Files\\ELSTAT-demographic-economic.csv") 
municipal_communities_df.rename(columns={"ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ": "Municipal_Community"}, inplace=True)
print(municipal_communities_df.shape)
print(municipal_communities_df.columns)
municipal_communities_df.head()

(76, 23)
Index(['Unnamed: 0', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ', 'ΔΗΜΟΣ', 'ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ',
       'ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ ΕΛΛ', 'Municipal_Community', 'Population',
       'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct', 'Διαζευγμένοι_pct',
       'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct', 'low_education_pct',
       'medium_education_pct', 'high_education_pct', 'unemployment_rate',
       'labor_force_participation_rate', 'primary_sector_pct',
       'secondary_sector_pct', 'tertiary_sector_pct', 'Area_km2'],
      dtype='object')


,Unnamed: 0,ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ,ΔΗΜΟΣ,ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ,ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ ΕΛΛ,Municipal_Community,Population,Άγαμοι_pct,Έγγαμοι_pct,Χήροι_pct,...,age_65_plus_pct,low_education_pct,medium_education_pct,high_education_pct,unemployment_rate,labor_force_participation_rate,primary_sector_pct,secondary_sector_pct,tertiary_sector_pct,Area_km2
0,0,ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΜΑΓΝΗΣΙΑΣ,ΔΗΜΟΣ ΒΟΛΟΥ,ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ ΒΟΛΟΥ,Δημοτική Κοινότητα Βόλου,Municipal Community of Volos,85806,0.4032,0.4630,0.0809,...,0.2221,0.3634,0.3663,0.2704,0.1545,0.4188,0.0313,0.1666,0.8022,26.790807
1,1,ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΜΑΓΝΗΣΙΑΣ,ΔΗΜΟΣ ΒΟΛΟΥ,ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ ΑΓΡΙΑΣ,Δημοτική Κοινότητα Αγριάς,Municipal Community of Agria,4926,0.3577,0.5173,0.0796,...,0.2359,0.4670,0.3341,0.1987,0.1555,0.3955,0.0816,0.1864,0.7314,3.737347
2,2,ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΜΑΓΝΗΣΙΑΣ,ΔΗΜΟΣ ΒΟΛΟΥ,ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ ΑΓΡΙΑΣ,Δημοτική Κοινότητα Δρακείας,Municipal Community of Drakeia,369,0.3496,0.4580,0.1382,...,0.2873,0.6667,0.2493,0.0840,0.1439,0.3767,0.4545,0.0826,0.4628,21.683415
3,3,ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΜΑΓΝΗΣΙΑΣ,ΔΗΜΟΣ ΒΟΛΟΥ,ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ ΑΙΣΩΝΙΑΣ,Δημοτική Κοινότητα Διμηνίου,Municipal Community of Dimini,2101,0.3889,0.5007,0.0776,...,0.2080,0.5437,0.3163,0.1395,0.2211,0.4327,0.1161,0.2365,0.6459,37.323451
4,4,ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΜΑΓΝΗΣΙΑΣ,ΔΗΜΟΣ ΒΟΛΟΥ,ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ ΑΙΣΩΝΙΑΣ,Δημοτική Κοινότητα Σέσκλου,Municipal Community of Sesklos,899,0.3960,0.5184,0.0645,...,0.2147,0.6009,0.2812,0.1179,0.1304,0.3838,0.1424,0.3609,0.4967,37.329725


## ***Create the model Dataset***

### ***Create all possible (NACE, Neighborhood) pairs***

In [60]:
all_naces = top3_df["NACE_Code"].unique()
all_neighborhoods = neighborhoods_df["Neighborhood"].unique()
print(f"{len(all_naces)} x {len(all_neighborhoods)} = {len(all_naces)*len(all_neighborhoods)}")

306 x 48 = 14688


In [61]:
pairs = pd.DataFrame(list(product(all_naces, all_neighborhoods)), columns=["NACE_Code", "Neighborhood"])

In [62]:
print(pairs.shape)
print(pairs.columns)
pairs.head()

(14688, 2)
Index(['NACE_Code', 'Neighborhood'], dtype='object')


,NACE_Code,Neighborhood
0,1.13,Dimini
1,1.13,Xrisi Akti Panagias
2,1.13,Sesklo
3,1.13,Agioi Anargiroi
4,1.13,Aivaliotika


### ***Create the Binary Label***

In [63]:
top3_lookup = top3_df.set_index("NACE_Code")[["Top1", "Top2", "Top3"]].to_dict("index")

In [64]:
pairs["IsTop3"] = pairs.apply(lambda row: is_top3(row["NACE_Code"], row["Neighborhood"], top3_lookup), axis=1)

In [35]:
print(pairs.shape)
print(pairs.columns)
print(pairs['IsTop3'].value_counts())
pairs.head()

(14688, 3)
Index(['NACE_Code', 'Neighborhood', 'IsTop3'], dtype='object')
IsTop3
0    13770
1      918
Name: count, dtype: int64


,NACE_Code,Neighborhood,IsTop3
0,1.13,Dimini,0
1,1.13,Xrisi Akti Panagias,0
2,1.13,Sesklo,0
3,1.13,Agioi Anargiroi,0
4,1.13,Aivaliotika,0


### ***Merge with: Neighborhood-level Features***

In [65]:
merged = pairs.merge(neighborhoods_df, on="Neighborhood", how="left")

In [66]:
merged = merged.drop(columns=['Neighborhood_Greek', 'Geometry'])

In [67]:
print(merged.shape)
print(merged.columns)
merged.head()

(14688, 12)
Index(['NACE_Code', 'Neighborhood', 'IsTop3', 'Municipal_Community',
       'Centroid_x', 'Centroid_y', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')


,NACE_Code,Neighborhood,IsTop3,Municipal_Community,Centroid_x,Centroid_y,Neighborhood_Area_km2,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km
0,1.13,Dimini,0,Municipal Community of Dimini,22.882945,39.349112,37.344776,6.154015,5.418577,0.443434,1.569912,4.273031
1,1.13,Xrisi Akti Panagias,0,Municipal Community of Sesklos,22.836899,39.305430,9.983032,11.932475,11.015584,2.299921,2.410375,10.100771
2,1.13,Sesklo,0,Municipal Community of Sesklos,22.838168,39.353051,27.367312,9.814666,9.191001,0.482308,4.965561,7.989610
3,1.13,Agioi Anargiroi,0,Municipal Community of Volos,22.924059,39.366937,0.774436,2.296184,1.978856,0.058142,0.063879,0.866021
4,1.13,Aivaliotika,0,Municipal Community of Volos,22.922660,39.343696,4.848904,3.508166,2.529372,0.224486,0.246542,1.913456


### ***Merge with: Municipal-Community-level Features***

In [68]:
merged = merged.merge(municipal_communities_df, on="Municipal_Community", how="left")

In [69]:
merged = merged.drop(columns=['Unnamed: 0', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ', 'ΔΗΜΟΣ', 'ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ', 'ΔΗΜΟΤΙΚΗ ΚΟΙΝΟΤΗΤΑ ΕΛΛ'])

### ***Encode the NACE code***

In [72]:
encoder = LabelEncoder()
merged["NACE_Label"] = encoder.fit_transform(merged["NACE_Code"])

In [73]:
print(merged.shape)
print(merged.columns)
merged.head()

(14688, 30)
Index(['NACE_Code', 'Neighborhood', 'IsTop3', 'Municipal_Community',
       'Centroid_x', 'Centroid_y', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km',
       'Population', 'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct',
       'Διαζευγμένοι_pct', 'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct',
       'low_education_pct', 'medium_education_pct', 'high_education_pct',
       'unemployment_rate', 'labor_force_participation_rate',
       'primary_sector_pct', 'secondary_sector_pct', 'tertiary_sector_pct',
       'Area_km2', 'NACE_Label'],
      dtype='object')


,NACE_Code,Neighborhood,IsTop3,Municipal_Community,Centroid_x,Centroid_y,Neighborhood_Area_km2,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,...,low_education_pct,medium_education_pct,high_education_pct,unemployment_rate,labor_force_participation_rate,primary_sector_pct,secondary_sector_pct,tertiary_sector_pct,Area_km2,NACE_Label
0,1.13,Dimini,0,Municipal Community of Dimini,22.882945,39.349112,37.344776,6.154015,5.418577,0.443434,...,0.5437,0.3163,0.1395,0.2211,0.4327,0.1161,0.2365,0.6459,37.323451,0
1,1.13,Xrisi Akti Panagias,0,Municipal Community of Sesklos,22.836899,39.305430,9.983032,11.932475,11.015584,2.299921,...,0.6009,0.2812,0.1179,0.1304,0.3838,0.1424,0.3609,0.4967,37.329725,0
2,1.13,Sesklo,0,Municipal Community of Sesklos,22.838168,39.353051,27.367312,9.814666,9.191001,0.482308,...,0.6009,0.2812,0.1179,0.1304,0.3838,0.1424,0.3609,0.4967,37.329725,0
3,1.13,Agioi Anargiroi,0,Municipal Community of Volos,22.924059,39.366937,0.774436,2.296184,1.978856,0.058142,...,0.3634,0.3663,0.2704,0.1545,0.4188,0.0313,0.1666,0.8022,26.790807,0
4,1.13,Aivaliotika,0,Municipal Community of Volos,22.922660,39.343696,4.848904,3.508166,2.529372,0.224486,...,0.3634,0.3663,0.2704,0.1545,0.4188,0.0313,0.1666,0.8022,26.790807,0


## ***Save the Data***

In [74]:
merged.to_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\7. LLM - Fine Tuning\\1. Fine-Tuning - Inference Datasets Creation\\2. Feature Importance Inspection\\Extracted CSV Files\\ml_model_dataset.csv", index=False)

In [77]:
joblib.dump(encoder, "C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\7. LLM - Fine Tuning\\1. Fine-Tuning - Inference Datasets Creation\\2. Feature Importance Inspection\\ML Model\\nace_label_encoder.pkl")

['C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\7. LLM - Fine Tuning\\1. Fine-Tuning - Inference Datasets Creation\\2. Feature Importance Inspection\\ML Model\\nace_label_encoder.pkl']